In [7]:
import pandas as pd
import pyodbc
import os
import io
# import smtplib
import paramiko
# from ftplib import FTP
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
# def ConnectFTP(server, username, password):
#     """
#     Establece una conexión FTP en modo pasivo y binario.
#     """
#     ftp = FTP(timeout=60)                 
#     ftp.connect(server, 21)               
#     ftp.login(user=username, passwd=password)

#     ftp.set_pasv(True)                    
#     ftp.voidcmd("TYPE I")                 

#     print(f"Conectado a {server}")
#     return ftp

# def UploadCsvToFTP(df, path):
#     """
#     Convierte un DataFrame a CSV en memoria y lo sube al FTP.
#     """
#     csv_buffer = io.BytesIO()
#     df.to_csv(csv_buffer, index=False, encoding='utf-8')
#     csv_buffer.seek(0)
#     ftp = ConnectFTP(
#         os.getenv('ftp_server'),
#         os.getenv('ftp_user'),
#         os.getenv('ftp_password')
#     )
#     remote_path = f"/pruebas/csv/{path}"
#     ftp.storbinary(f"STOR {remote_path}", csv_buffer)
#     ftp.quit()
#     print("Archivo subido correctamente:", remote_path)

def ConnectSFTP(server, username, password, port=22):
    """
    Establece una conexión SFTP mediante SSH.

    Retorna:
        ssh: cliente SSH necesario para cerrar correctamente la conexión.
        sftp: cliente utilizado para navegar y transferir archivos.
    """
    ssh = paramiko.SSHClient()
    ssh.load_system_host_keys()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    ssh.connect(
        hostname=server,
        port=port,
        username=username,
        password=password,
        timeout=60,
    )

    sftp = ssh.open_sftp()

    print(f"Conectado mediante SFTP a {server}:{port}")
    return ssh, sftp

def UploadCsvToSFTP(df, path):
    """
    Convierte un DataFrame a CSV en memoria y lo sube mediante SFTP.
    """
    csv_buffer = io.BytesIO()
    df.to_csv(
        csv_buffer,
        index=False,
        encoding="utf-8"
    )
    csv_buffer.seek(0)
    ssh, sftp = ConnectSFTP(
        os.getenv("ftp_server"),
        os.getenv("ftp_user"),
        os.getenv("ftp_password")
    )
    remote_path = f"riecs/pruebas2/csv/{path}"
    try:
        sftp.putfo(csv_buffer, remote_path)
        print("Archivo subido correctamente:", remote_path)
    finally:
        sftp.close()
        ssh.close()

def QuerySQL(conn, table):
    '''
    Descripcion: Esta funcion ejecuta una consulta SQL en la base de datos.
    Parametros:
    conn (Connection): La conexion a la base de datos.
    table (str): El nombre de la tabla a consultar.
    Retorna: Un DataFrame con los resultados de la consulta.
    '''
    current_dir = os.getcwd()    
    sql_file_path = os.path.join(current_dir, 'SQL', table)
    with open(sql_file_path, 'r') as file:
        query = file.read()
    return pd.read_sql(query, conn)

def SQLConnection(table):
    '''
    Descripcion: Esta funcion establece una conexion a la base de datos utilizando las credenciales almacenadas en un archivo .env.
    Parametros:
    table (str): El nombre de la tabla a consultar.
    Retorna: DataFrame de pandas con los resultados de la consulta ejecutada.
    '''
    conn = pyodbc.connect(
        f'DRIVER={os.getenv("DRIVER")};'
        f'SERVER={os.getenv("SERVER")};'
        f'DATABASE={os.getenv("DATABASE")};'
        f'UID={os.getenv("UID")};'
        f'PWD={os.getenv("PWD")};'
        f'TrustServerCertificate={os.getenv("TrustServerCertificate")};'
)
    df = QuerySQL(conn, table)
    conn.close()
    return df

def PivotCombinadas(df):
    '''
    Descripcion: Esta funcion pivota las columnas de sustancias combinadas en el DataFrame, creando nuevas columnas para cada sustancia combinada asociada a un FolioId.
    Parametros:
    df (pd.DataFrame): El DataFrame que contiene los datos de sustancias combinadas.
    list (list): Lista de columnas que se conservarán en el DataFrame final.
    Retorna: DataFrame con las columnas de sustancias combinadas pivotadas y las columnas especificadas en list.
    '''
    df['SustanciaComb_n'] = df.groupby('FolioId').cumcount() + 1
    df_pivot_comb = df.pivot(index = 'FolioId', columns= 'SustanciaComb_n', values='EISC_SustanciaId' )
    df_pivot_comb.columns = [f'SustanciaComb{int(col)}' for col in df_pivot_comb.columns]
    df_final = df_pivot_comb.reset_index()
    return df_final

def PivotMotivo(df):
    '''
    Descripcion: Esta funcion pivota las columnas de motivos en el DataFrame, creando nuevas columnas para cada motivo asociado a un FolioId.
    Parametros:
    df (pd.DataFrame): El DataFrame que contiene los datos de motivos.
    list (list): Lista de columnas que se conservarán en el DataFrame final.
    Retorna: DataFrame con las columnas de motivos pivotadas y las columnas especificadas en list.
    '''
    df['Motivo_n'] = df.groupby('FolioId').cumcount() + 1
    df_pivot_motivo = df.pivot(index = 'FolioId', columns= 'Motivo_n', values='MotivoConsultaId' )
    df_pivot_motivo.columns = [f'Motivo_{int(col)}' for col in df_pivot_motivo.columns]
    df_final = df_pivot_motivo.reset_index()
    return df_final

def PivotSustancias(df, order_col=None):
    '''
    Descripcion: Esta funcion pivota las columnas de sustancias en el DataFrame, creando nuevas columnas para cada sustancia asociada a un FolioId.
    Parametros:
    df (pd.DataFrame): El DataFrame que contiene los datos de sustancias.
    order_col (str, opcional): El nombre de la columna por la cual ordenar antes de pivotar. Si no se proporciona, no se ordenará.
    Retorna: DataFrame con las columnas de sustancias pivotadas.
    '''
    if order_col:
        df = df.sort_values(['FolioId', order_col])

    df = df.copy()
    df['SustanciaId_n'] = df.groupby('FolioId').cumcount() + 1

    value_cols = [
        'SustanciaId',
        'EdadInicio',
        'OrdenConsumo',
        'ComunPrevalenciaId',
        'ComunPrimeraFormaAdministracionId',
        'ComunSegundaFormaAdministracionId',
        'ComunTerceraFormaAdministracionId',
        'ComunAbstinenciaId',
        'ComunUltimoConsumoId',
        'Dosis'
    ]

    pivots = []
    for col in value_cols:
        p = df.pivot(index='FolioId', columns='SustanciaId_n', values=col)
        p.columns = [f'{col}{int(i)}' for i in p.columns]
        pivots.append(p)

    df_final = pivots[0]
    for p in pivots[1:]:
        df_final = df_final.join(p)

    return df_final.reset_index()

def MergeAndPush(df_motivo, df_sustancias, df_combinadas, df_paciente):
    '''
    Descripcion: Esta funcion une varios DataFrames en uno solo utilizando la columna 'FolioId' como clave de union.
    Parametros:
    df_motivo (pd.DataFrame): DataFrame que contiene los datos de motivos.
    df_sustancias (pd.DataFrame): DataFrame que contiene los datos de sustancias.
    df_combinadas (pd.DataFrame): DataFrame que contiene los datos de sustancias combinadas.
    df_paciente (pd.DataFrame): DataFrame que contiene los datos de pacientes.
    Retorna: DataFrame resultante de la union de los DataFrames proporcionados.
    '''
    df_join = df_paciente.merge(df_motivo, on='FolioId', how='left')
    df_join = df_join.merge(df_sustancias, on='FolioId', how='left')
    df_join = df_join.merge(df_combinadas, on='FolioId', how='left')
    return df_join

In [9]:
def main():
    try:
        processors = {'Motivo.sql': PivotMotivo, 'Sustancias.sql': PivotSustancias, 'Combinadas.sql': PivotCombinadas}
        results = {}
        for table, func in processors.items():
            df = SQLConnection(table)
            df_processed = func(df)
            results[table] = df_processed
    except Exception as e:
        print("1)Extract data / Error durante el procesamiento de datos:", e)
        return 
    
    try:
        df_join = MergeAndPush(results['Motivo.sql'], results['Sustancias.sql'], results['Combinadas.sql'], SQLConnection('Paciente.sql'))
        UploadCsvToSFTP(df_join, 'EntrevistaInicial/EntrevisaInicial(25).csv')
        return df_join 
    except Exception as e:
        print("2)Merge data / Error durante la union o subida de datos:", e)
        return
    
if __name__ == "__main__":
    df_join = main()

C:\Users\franc\AppData\Local\Temp\ipykernel_12932\1648960930.py:93: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)
C:\Users\franc\AppData\Local\Temp\ipykernel_12932\1648960930.py:93: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)
C:\Users\franc\AppData\Local\Temp\ipykernel_12932\1648960930.py:93: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)
C:\Users\franc\AppData\Local\Temp\ipykernel_12932\1648960930.py:93: UserWarning: pandas onl

Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/EntrevistaInicial/EntrevisaInicial(25).csv
